In [1]:
import zipfile

with zipfile.ZipFile("Train.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

with zipfile.ZipFile("Test.zip", 'r') as zip_ref:
    zip_ref.extractall("data")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [3]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


train_dataset = datasets.ImageFolder("data/Train/Train", transform=transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print(train_dataset.classes)  

['Jade', 'James', 'Jane', 'Joel', 'Jovi']


In [14]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3),   # (64x64 → 62x62)
            nn.ReLU(),
            nn.MaxPool2d(2),       # (62 → 31)

            nn.Conv2d(32, 64, 3),
            nn.ReLU(),
            nn.MaxPool2d(2)        # (29 → 14)
        )

        =.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 14 * 14, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 5)   # 5 classes
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

model = CNN()

In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [16]:
epochs = 15

for epoch in range(epochs):
    running_loss = 0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}")

Epoch 1, Loss: 73.9460
Epoch 2, Loss: 32.8948
Epoch 3, Loss: 26.4665
Epoch 4, Loss: 25.8338
Epoch 5, Loss: 23.1225
Epoch 6, Loss: 20.5175
Epoch 7, Loss: 19.0390
Epoch 8, Loss: 19.3398
Epoch 9, Loss: 16.8660
Epoch 10, Loss: 17.2452
Epoch 11, Loss: 16.0880
Epoch 12, Loss: 14.6898
Epoch 13, Loss: 13.9200
Epoch 14, Loss: 12.3791
Epoch 15, Loss: 12.9040


In [17]:
from PIL import Image

def predict(image_path):
    model.eval()   # important

    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)

    return train_dataset.classes[predicted.item()]

In [18]:
import os 

print(os.listdir("data/Test/Test/Jane"))

['-601.png', '-602.png', '-603.png', '-604.png', '-605.png', '-606.png', '-607.png', '-608.png', '-609.png', '-610.png', '-611.png', '-612.png', '-613.png', '-614.png', '-615.png', '-616.png', '-617.png', '-618.png', '-619.png', '-620.png', '-621.png', '-622.png', '-623.png', '-624.png', '-625.png', '-626.png', '-627.png', '-628.png', '-629.png', '-630.png', '-631.png', '-632.png', '-633.png', '-634.png', '-635.png', '-636.png', '-637.png', '-638.png', '-639.png', '-640.png', '-641.png', '-642.png', '-643.png', '-644.png', '-645.png', '-646.png', '-647.png', '-648.png', '-649.png', '-650.png', '-651.png', '-652.png', '-653.png', '-654.png', '-655.png', '-656.png', '-657.png', '-658.png', '-659.png', '-660.png', '-661.png', '-662.png', '-663.png', '-664.png', '-665.png', '-666.png', '-667.png', '-668.png', '-669.png', '-670.png', '-671.png', '-672.png', '-673.png', '-674.png', '-675.png', '-676.png', '-677.png', '-678.png', '-679.png', '-680.png', '-681.png', '-682.png', '-683.png', '-6

In [19]:
print(predict("data/Test/Test/Jane/-601.png"))

Jane


In [20]:
print(predict("data/Test/Test/Jane/-781.png"))

Jane


In [21]:
import os

folder = "data/Test/Test/Jane"

for img_name in os.listdir(folder)[:5]:   # first 5 images
    img_path = os.path.join(folder, img_name)

    print("Image:", img_name)
    print("Prediction:", predict(img_path))
    print("-"*20)

Image: -601.png
Prediction: Jane
--------------------
Image: -602.png
Prediction: Jane
--------------------
Image: -603.png
Prediction: Jane
--------------------
Image: -604.png
Prediction: Jane
--------------------
Image: -605.png
Prediction: Jane
--------------------


In [22]:
folders = ["Jade", "James", "Joel", "Jovi"]

for person in folders:
    folder = f"data/Test/Test/{person}"

    print(f"\nTesting {person}")
    
    for img_name in os.listdir(folder)[:3]:
        img_path = os.path.join(folder, img_name)

        print("Actual:", person)
        print("Predicted:", predict(img_path))
        print("-"*20)


Testing Jade
Actual: Jade
Predicted: Jade
--------------------
Actual: Jade
Predicted: Jade
--------------------
Actual: Jade
Predicted: Jade
--------------------

Testing James
Actual: James
Predicted: Joel
--------------------
Actual: James
Predicted: James
--------------------
Actual: James
Predicted: James
--------------------

Testing Joel
Actual: Joel
Predicted: Joel
--------------------
Actual: Joel
Predicted: Joel
--------------------
Actual: Joel
Predicted: Joel
--------------------

Testing Jovi
Actual: Jovi
Predicted: Jovi
--------------------
Actual: Jovi
Predicted: Jovi
--------------------
Actual: Jovi
Predicted: Jovi
--------------------


In [23]:
correct = 0
total = 0

test_dir = "data/Test/Test"

for person in os.listdir(test_dir):
    folder = os.path.join(test_dir, person)

    for img_name in os.listdir(folder):
        img_path = os.path.join(folder, img_name)

        pred = predict(img_path)

        if pred == person:
            correct += 1

        total += 1

print("Accuracy:", correct/total)

Accuracy: 0.957
